# EDA — LLM Agentic Legal Information Retrieval

> Inspired by [Hybrid Legal Retrieval Baseline](https://www.kaggle.com/code/ramazanturann/hybrid-legal-retrieval-baseline) by ramazanturann.

> Assisted by Claude

Swiss law is notoriously precise. Every word in every article matters. So before we throw models at this problem, let's actually understand what we're working with — the data, the quirks, and where the real difficulty lives.

This notebook covers:
- What are the queries, really?
- How many citations per query and why does it vary so wildly?
- Are we even truncating queries correctly?
- What's inside the two corpora — and are the right answers actually findable?

If this notebook helps you, please upvote — it keeps me going!

---

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from transformers import AutoTokenizer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')
plt.rcParams['figure.dpi'] = 110

# Works both on Kaggle and locally
try:
    BASE = '/kaggle/input/llm-agentic-legal-information-retrieval'
    open(f'{BASE}/train.csv')
except FileNotFoundError:
    BASE = '../data'

CHAR_CUTOFF    = 2000  # baseline [:text_truncate]
EMBEDDER_LIMIT = 8192  # bge-m3 max tokens
RERANKER_LIMIT = 512   # bge-reranker-v2-m3 max tokens

In [ ]:
train_df = pd.read_csv(f'{BASE}/train.csv')
val_df   = pd.read_csv(f'{BASE}/val.csv')
test_df  = pd.read_csv(f'{BASE}/test.csv')
laws_df  = pd.read_csv(f'{BASE}/laws_de.csv')
court_df = pd.read_csv(f'{BASE}/court_considerations.csv', usecols=['citation', 'text'])

laws_df['text']  = laws_df['text'].fillna('')
court_df['text'] = court_df['text'].fillna('')

print(f'train  : {len(train_df):>10,} rows — {list(train_df.columns)}')
print(f'val    : {len(val_df):>10,} rows — {list(val_df.columns)}')
print(f'test   : {len(test_df):>10,} rows — {list(test_df.columns)}')
print(f'laws   : {len(laws_df):>10,} rows — {list(laws_df.columns)}')
print(f'court  : {len(court_df):>10,} rows — {list(court_df.columns)}')

## What are the queries, actually?

Before plotting anything, let's just read some examples. A lot of intuition gets built this way.

In [ ]:
train_df['n_citations'] = train_df['gold_citations'].str.split(';').str.len()

print('A train query (German):')
print('─' * 60)
row = train_df[train_df['n_citations'] == 1].sample(1, random_state=7).iloc[0]
print(f"Answer : {row['gold_citations']}")
print(f"Query  : {row['query'][:400]}")

print()
print('A val query (English):')
print('─' * 60)
print(val_df['query'].iloc[0][:400])

print()
print('A complex train query (38 citations):')
print('─' * 60)
row = train_df[train_df['n_citations'] >= 38].sample(1, random_state=2).iloc[0]
print(f"Answer : {row['n_citations']} citations — {row['gold_citations'][:200]}...")
print(f"Query  : {row['query'][:400]}...")

These are not keyword searches. A simple query is a focused legal question with a clear answer. A complex query is a full *Sachverhalt* — a multi-page case scenario covering inheritance, property registration, financial instruments — each aspect needing its own article of law.

Notice something important: **train queries are in German, but val and test queries are in English**. The corpus is in German (plus French and Italian for court decisions). So the actual task at eval time is cross-lingual — English queries matched against a German corpus. Train is in German, which makes it useful for learning retrieval patterns but doesn't fully represent the eval distribution.

## How many citations per query?

Let's look at the distribution. The spread here has huge implications for how we design the retrieval system.

In [ ]:
val_df['n_citations'] = val_df['gold_citations'].str.split(';').str.len()

f, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].hist(train_df['n_citations'], bins=range(1, 46), color='steelblue', edgecolor='white')
axes[0].axvline(10, color='crimson', linestyle='--', linewidth=2, label='baseline top_k_final=10')
axes[0].set_xlabel('Citations per query')
axes[0].set_ylabel('Number of queries')
axes[0].set_title('Gold citation count (train)')
axes[0].legend()

for df, label, color in [(train_df, 'train', 'steelblue'), (val_df, 'val', 'darkorange')]:
    s = np.sort(df['n_citations'].values)
    axes[1].plot(s, np.linspace(0, 1, len(s)), label=label, linewidth=2.5, color=color)
axes[1].axvline(10, color='crimson', linestyle='--', linewidth=2, label='baseline top_k_final=10')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_xlabel('Citations per query')
axes[1].set_title('CDF — train vs val')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Train — median: {train_df['n_citations'].median():.0f}, max: {train_df['n_citations'].max()}")
print(f"Val   — median: {val_df['n_citations'].median():.0f}, max: {val_df['n_citations'].max()}")

The baseline caps predictions at `top_k_final=10`. For val, the median is 22 citations per query. That means the system structurally cannot achieve full recall on most val queries — the correct answers aren't even in the candidate set.

But the other direction hurts too. Returning 10 results for a 1-citation query means 9 false positives. The metric is **Macro F1** — even with perfect recall, F1 would be around 18%. You need to return roughly the *right number* of citations, not just a fixed count.

Observations:
1. The majority of train queries need only 1–5 citations (simple focused questions).
2. A long tail of complex Sachverhalt cases need 20–44 citations.
3. Val skews heavily toward the complex end — median 22 vs 2 in train.
4. A fixed top-k fails at both ends. An adaptive cutoff based on score gaps is the right approach.

## Are we even sending the full query to the model?

The baseline truncates every query at 2,000 characters before encoding. Let's see how much we're throwing away.

In [ ]:
train_df['query_len'] = train_df['query'].str.len()

lens = np.sort(train_df['query_len'].values)

f, ax = plt.subplots(figsize=(12, 5))
ax.plot(lens, np.linspace(0, 1, len(lens)), color='steelblue', linewidth=2.5)
ax.axvline(CHAR_CUTOFF, color='crimson', linestyle='--', linewidth=2,
           label=f'Baseline cutoff ({CHAR_CUTOFF:,} chars)')
ax.fill_betweenx([0, 1], 0, CHAR_CUTOFF, alpha=0.06, color='green')
ax.fill_betweenx([0, 1], CHAR_CUTOFF, 10000, alpha=0.06, color='red')
ax.set_xlim(0, 10000)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('Query length (characters)')
ax.set_ylabel('Cumulative % of queries')
ax.set_title('Query length CDF — how much does the baseline cut off?')
ax.legend()
plt.tight_layout()
plt.show()

for c in [1000, 2000, 3000, 5000]:
    print(f'<= {c:>5} chars: {(train_df["query_len"] <= c).mean()*100:.1f}% of queries fully captured')

n_trunc = (train_df['query_len'] == 32767).sum()
if n_trunc:
    print(f'\nNote: {n_trunc} queries are exactly 32,767 chars (2^15−1) — CSV cell truncation artifact, not real data')

The baseline silently drops the tail of **22% of all queries** mid-sentence. These are precisely the complex multi-part cases where the full context matters most.

Here's the surprising part: **bge-m3 supports up to 8,192 tokens**, not 512. At a typical German characters-per-token ratio of ~3–4, that's roughly 25,000 characters of capacity. The 2,000 char cutoff uses less than 10% of what the model can actually handle.

## What's inside the two corpora?

Laws and court are very different datasets. Let's look at the text lengths and what a typical row actually contains.

In [ ]:
laws_df['char_len']  = laws_df['text'].str.len()
court_df['char_len'] = court_df['text'].str.len()

print('Sample law article:')
row = laws_df.sample(1, random_state=5).iloc[0]
print(f"  citation : {row['citation']}")
print(f"  text     : {row['text']}")

print()
print('Sample court consideration:')
row = court_df[court_df['char_len'].between(200, 400)].sample(1, random_state=5).iloc[0]
print(f"  citation : {row['citation']}")
print(f"  text     : {row['text']}")

print()
print('Some very short court rows (noise):')
for t in court_df[court_df['char_len'] < 50]['text'].sample(4, random_state=1).values:
    print(f"  {repr(t)}")

In [ ]:
f, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, df, label, color in [
    (axes[0], laws_df,  'Laws',  'steelblue'),
    (axes[1], court_df, 'Court', 'darkorange')
]:
    clipped = df['char_len'].clip(upper=df['char_len'].quantile(0.99))
    ax.hist(clipped, bins=60, color=color, edgecolor='white')
    ax.axvline(CHAR_CUTOFF, color='crimson', linestyle='--', linewidth=2,
               label=f'Baseline cutoff ({CHAR_CUTOFF})')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} — char length (clipped p99)')
    ax.legend()

plt.tight_layout()
plt.show()

print(f"{'':8} {'rows':>10}  {'median chars':>14}  {'> 2000 chars':>14}  {'< 100 chars':>12}")
print('─' * 65)
for df, label in [(laws_df, 'laws'), (court_df, 'court')]:
    print(f"{label:<8} {len(df):>10,}  "
          f"{df['char_len'].median():>14.0f}  "
          f"{(df['char_len'] > CHAR_CUTOFF).mean()*100:>13.1f}%  "
          f"{(df['char_len'] < 100).mean()*100:>11.1f}%")

A few things jump out:

1. Law articles are short — median around 200 characters. Most fit well within any reasonable token budget. Only a small fraction exceed 2,000 chars.
2. Court is not a collection of full decisions. Each row is one *consideration* or section of a ruling, already pre-chunked. The file is 2.5M rows not because each text is long but because there are a huge number of individual sections.
3. **17.5% of court rows are under 100 characters** — procedural one-liners: *"Il est statué sans frais"*, *"Auf die Beschwerde wird nicht eingetreten"*. Zero retrieval value. These pollute the BM25 index and waste embedding compute. Filtering them before indexing is a free improvement.
4. The court corpus is multilingual — German, French, and Italian — reflecting Switzerland's linguistic regions.

## Token lengths — what does the model actually see?

Characters are a rough proxy. Let's tokenize with bge-m3's own tokenizer to measure the true token counts.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-m3')

def token_lengths(texts, batch_size=1024):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc='tokenizing', leave=False):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            enc = tokenizer(list(texts[i:i+batch_size]), truncation=False, add_special_tokens=True)
        out.extend(len(ids) for ids in enc['input_ids'])
    return np.array(out)

print('tokenizing laws...')
laws_df['token_len']  = token_lengths(laws_df['text'])
print('tokenizing court (may take a few minutes)...')
court_df['token_len'] = token_lengths(court_df['text'])
print('done')

In [ ]:
f, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, df, label, color in [
    (axes[0], laws_df,  'Laws',  'steelblue'),
    (axes[1], court_df, 'Court', 'darkorange')
]:
    clipped = df['token_len'].clip(upper=df['token_len'].quantile(0.99))
    ax.hist(clipped, bins=60, color=color, edgecolor='white')
    ax.axvline(RERANKER_LIMIT, color='purple', linestyle='--', linewidth=2,
               label=f'Reranker limit ({RERANKER_LIMIT} tok)')
    ax.axvline(EMBEDDER_LIMIT, color='crimson', linestyle='--', linewidth=2,
               label=f'Embedder limit ({EMBEDDER_LIMIT} tok)')
    ax.set_xlabel('Tokens')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} — token length (clipped p99)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"{'':8}  {'> reranker (512)':>18}  {'> embedder (8192)':>18}  {'chars/token':>12}  {'8192 tok ≈ X chars':>20}")
print('─' * 85)
for df, label in [(laws_df, 'laws'), (court_df, 'court')]:
    ratio = (df['char_len'] / df['token_len'].clip(lower=1)).median()
    print(f"{label:<8}  "
          f"{(df['token_len'] > RERANKER_LIMIT).mean()*100:>17.1f}%  "
          f"{(df['token_len'] > EMBEDDER_LIMIT).mean()*100:>17.1f}%  "
          f"{ratio:>12.2f}  "
          f"{EMBEDDER_LIMIT * ratio:>18,.0f}")

The corpus texts are cut to 2,000 characters before embedding, even though bge-m3 can handle ~25,000 characters at a typical German chars-per-token ratio. The reranker adds a second, harder cut at 512 tokens.

Observations:
1. Laws are mostly short — the 2,000 char cutoff and the 512-token reranker limit rarely bite them.
2. Court considerations have a long tail. A meaningful share exceed 512 tokens, so the reranker silently drops the back half of those texts. With no newlines to split on cleanly, chunking court is non-trivial.
3. Removing the 2,000 char corpus truncation is a nearly free improvement for laws and a meaningful one for long court rows.

## Are the correct answers actually in the corpus?

This is the most important sanity check. A retrieval system cannot find answers that aren't there.

In [ ]:
laws_cits  = set(laws_df['citation'].dropna())
court_cits = set(court_df['citation'].dropna())
all_cits   = laws_cits | court_cits

for split, df in [('train', train_df), ('val', val_df)]:
    gold     = df['gold_citations'].str.split(';').explode().str.strip().dropna()
    in_laws  = gold.isin(laws_cits).mean()  * 100
    in_court = gold.isin(court_cits).mean() * 100
    missing  = (~gold.isin(all_cits)).mean() * 100
    print(f'{split} ({len(gold):,} total citations)')
    print(f'  in laws corpus : {in_laws:.1f}%')
    print(f'  in court corpus: {in_court:.1f}%')
    print(f'  not in corpus  : {missing:.1f}%')
    print()

# What are the missing ones?
train_gold = train_df['gold_citations'].str.split(';').explode().str.strip().dropna()
missing    = train_gold[~train_gold.isin(all_cits)]
print('Most common missing citations:')
print(missing.value_counts().head(10).to_string())

28.8% of train gold citations are completely absent from the corpus — all law articles from codes like OR (contract law), ZGB (civil code), StGB (criminal code) that were left out of `laws_de.csv`.

This sounds alarming but look at val: **0% missing**. Every single correct answer in the validation set exists in the corpus. This is almost certainly deliberate — the competition organizers curated val and test to only reference laws that are present, meaning the training missing rate is a data generation artifact, not a ceiling on your leaderboard score.

One more thing worth noting: only 1.2% of correct answers are court citations. The baseline skips dense retrieval on court entirely (it's 2.5M rows — expensive). That turns out to be nearly costless for recall.

## What we learned

| Finding | Impact | Action |
|---------|--------|--------|
| `top_k_final=10` misses most val queries — median is 22 citations | High | Raise top-k; build adaptive score-gap cutoff |
| Fixed top-k tanks F1 on simple queries (9 false positives for a 1-citation query) | High | Adaptive top-k is essential |
| Baseline truncates 22% of queries; bge-m3 supports 8,192 tokens not 512 | Medium | Raise char cutoff to at least 5,000 |
| 17.5% of court rows are procedural noise under 100 chars | Medium | Filter before indexing |
| 28.8% of train labels missing from corpus — val/test are 100% covered | Low | Expected noise; ignore for LB |

The two highest-leverage improvements are both about `top_k`: raising the ceiling and making it adaptive. Everything else is secondary.